# CW2 Part B — Sparse GP Preference Learning with Bayesian Optimisation

>  **FOR GOOGLE COLAB USE ONLY**
> This notebook is configured for Google Colab with GPU acceleration.
> It installs specific JAX CUDA packages and assumes a Colab GPU runtime.
> **Do not run this notebook locally** — use `bo_loop.py` or `bo_loop_with_checks.py` instead.
>
> Before running: **Runtime → Change runtime type → T4 GPU**

In [ ]:
# Install JAX with CUDA 12 support and required packages
!pip install -q -U "jax[cuda12]" optax

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
import jax.numpy as jnp
import jax.scipy as jsp
import optax
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Verify GPU is available
try:
    _device = jax.devices("cuda")[0]
    print(f"Using device: {_device}")
except RuntimeError:
    raise RuntimeError(
        "No CUDA device found. Go to Runtime > Change runtime type > GPU."
    )

## Data Generation

In [ ]:
def latent_function(x):
    x0 = x[..., 0]
    x1 = x[..., 1]
    background = (
        0.8 * jnp.sin(2.0 * x0 * x1) +
        0.6 * jnp.cos(3.0 * x0) +
        0.6 * jnp.sin(3.0 * x1) +
        0.5 * jnp.sin(1.5 * (x0**2 + x1**2))
    )
    dominant_peak = 4 * jnp.exp(-1.0 * ((x0 - 1.2)**2 + (x1 + 1.0)**2))
    return background + dominant_peak


def generate_initial_preference_data(num_points=5, num_pairs=25, noise_std=0.5, seed=42):
    rng = jax.random.PRNGKey(seed)
    X = jax.random.uniform(rng, shape=(num_points, 2), minval=-3, maxval=3)
    latent_values = latent_function(X)
    rng, rng_subkey = jax.random.split(rng)
    indices_i = jax.random.randint(rng_subkey, shape=(num_pairs,), minval=0, maxval=num_points)
    rng, rng_subkey = jax.random.split(rng)
    indices_j = jax.random.randint(rng_subkey, shape=(num_pairs,), minval=0, maxval=num_points)
    X_i = X[indices_i]
    X_j = X[indices_j]
    latent_diff = latent_values[indices_i] - latent_values[indices_j]
    rng, rng_subkey = jax.random.split(rng)
    noise = noise_std * jax.random.normal(rng_subkey, shape=latent_diff.shape)
    y_ij = (latent_diff + noise > 0).astype(jnp.float32)
    return X_i, X_j, y_ij, X, latent_values


def sample_preferences(rng_key, candidate_1, candidate_2, preference_samples_per_location=25):
    latent_diff = latent_function(candidate_1) - latent_function(candidate_2)
    sigmoid_prob = jax.nn.sigmoid(latent_diff)
    rng_key, rng_subkey = jax.random.split(rng_key)
    preferences = jax.random.bernoulli(
        rng_subkey, p=sigmoid_prob, shape=(preference_samples_per_location,)
    ).astype(jnp.float32)
    new_X_i = jnp.tile(candidate_1, (preference_samples_per_location, 1))
    new_X_j = jnp.tile(candidate_2, (preference_samples_per_location, 1))
    return new_X_i, new_X_j, preferences

## Sparse GP Model

In [ ]:
def rbf_kernel(x1, x2, lengthscale, variance, diag=False):
    if x1.ndim == 1:
        x1 = x1.reshape(-1, 1)
        x2 = x2.reshape(-1, 1)
    if diag:
        dist_sq_diag = jnp.sum((x1 - x2) ** 2, axis=-1)
        return variance * jnp.exp(-0.5 * dist_sq_diag / (lengthscale ** 2))
    else:
        dist_sq = jnp.sum((x1[:, None, :] - x2[None, :, :]) ** 2, axis=-1)
        return variance * jnp.exp(-0.5 * dist_sq / (lengthscale ** 2))


def stable_cholesky(K, jitter=1e-5):
    return jnp.linalg.cholesky(K + jitter * jnp.eye(K.shape[0]))


def build_var_chol(var_chol_log_diag, var_chol_lower, M):
    L_v = jnp.zeros((M, M))
    lower_idx = jnp.tril_indices(M, k=-1)
    L_v = L_v.at[lower_idx].set(var_chol_lower)
    diag_idx = jnp.diag_indices(M)
    L_v = L_v.at[diag_idx].set(jnp.exp(var_chol_log_diag))
    return L_v


def svgp_predict(params, X_test, full_cov=False):
    ls  = jnp.exp(params["log_lengthscale"])
    var = jnp.exp(params["log_variance"])
    Z = params["inducing_inputs"]
    m = params["variational_mean"]
    M = Z.shape[0]
    L_v = build_var_chol(params["var_chol_log_diag"], params["var_chol_lower"], M)
    K_zz     = rbf_kernel(Z, Z, ls, var)
    L_zz     = stable_cholesky(K_zz)
    K_z_test = rbf_kernel(Z, X_test, ls, var)
    alpha    = jsp.linalg.solve_triangular(L_zz, K_z_test, lower=True)
    mu = alpha.T @ m
    B  = L_v.T @ alpha
    if full_cov:
        K_tt = rbf_kernel(X_test, X_test, ls, var)
        cov  = K_tt - alpha.T @ alpha + B.T @ B
    else:
        k_diag = rbf_kernel(X_test, X_test, ls, var, diag=True)
        cov    = k_diag - jnp.sum(alpha ** 2, axis=0) + jnp.sum(B ** 2, axis=0)
    return mu, cov


def svgp_kl_divergence(params):
    m = params["variational_mean"]
    M = m.shape[0]
    L_v = build_var_chol(params["var_chol_log_diag"], params["var_chol_lower"], M)
    trace_S   = jnp.sum(L_v ** 2)
    quad      = jnp.dot(m, m)
    log_det_S = 2.0 * jnp.sum(params["var_chol_log_diag"])
    return 0.5 * (trace_S + quad - M - log_det_S)


def svgp_ell_preference(params, X_i_batch, X_j_batch, y_ij_batch, N_total, rng_key, n_samples=20):
    ls  = jnp.exp(params["log_lengthscale"])
    var = jnp.exp(params["log_variance"])
    Z   = params["inducing_inputs"]
    m   = params["variational_mean"]
    M   = Z.shape[0]
    L_v  = build_var_chol(params["var_chol_log_diag"], params["var_chol_lower"], M)
    K_zz = rbf_kernel(Z, Z, ls, var)
    L_zz = stable_cholesky(K_zz)
    B_size = X_i_batch.shape[0]
    X_pairs   = jnp.concatenate([X_i_batch, X_j_batch], axis=0)
    K_z_pairs = rbf_kernel(Z, X_pairs, ls, var)
    alpha_all = jsp.linalg.solve_triangular(L_zz, K_z_pairs, lower=True)
    alpha_i = alpha_all[:, :B_size]
    alpha_j = alpha_all[:, B_size:]
    B_i     = L_v.T @ alpha_i
    B_j     = L_v.T @ alpha_j
    mu_i    = alpha_i.T @ m
    mu_j    = alpha_j.T @ m
    mu_diff = mu_i - mu_j
    k_ii = rbf_kernel(X_i_batch, X_i_batch, ls, var, diag=True)
    k_jj = rbf_kernel(X_j_batch, X_j_batch, ls, var, diag=True)
    k_ij = rbf_kernel(X_i_batch, X_j_batch, ls, var, diag=True)
    Sigma_ii = k_ii - jnp.sum(alpha_i ** 2, axis=0) + jnp.sum(B_i ** 2, axis=0)
    Sigma_jj = k_jj - jnp.sum(alpha_j ** 2, axis=0) + jnp.sum(B_j ** 2, axis=0)
    Sigma_ij = k_ij - jnp.sum(alpha_i * alpha_j, axis=0) + jnp.sum(B_i * B_j, axis=0)
    var_diff = jnp.clip(Sigma_ii + Sigma_jj - 2.0 * Sigma_ij, 1e-6)
    std_diff = jnp.sqrt(var_diff)
    rng_key, subkey = jax.random.split(rng_key)
    eps    = jax.random.normal(subkey, shape=(n_samples, B_size))
    f_diff = mu_diff[None, :] + std_diff[None, :] * eps
    y       = y_ij_batch[None, :].astype(jnp.float32)
    log_lik = y * (-jax.nn.softplus(-f_diff)) + (1.0 - y) * (-jax.nn.softplus(f_diff))
    ell = (N_total / B_size) * jnp.sum(jnp.mean(log_lik, axis=0))
    return ell, rng_key


def svgp_elbo_preference(params, X_i_batch, X_j_batch, y_ij_batch, N_total, rng_key, n_samples=20):
    ell, rng_key = svgp_ell_preference(
        params, X_i_batch, X_j_batch, y_ij_batch, N_total, rng_key, n_samples
    )
    kl = svgp_kl_divergence(params)
    return ell - kl, rng_key


def init_params_preference(num_inducing, rng_key):
    M = num_inducing
    rng_key, subkey = jax.random.split(rng_key)
    Z = jax.random.uniform(subkey, shape=(M, 2), minval=-3.0, maxval=3.0)
    return {
        "log_lengthscale":   jnp.array(0.0),
        "log_variance":      jnp.array(0.0),
        "inducing_inputs":   Z,
        "variational_mean":  jnp.zeros(M),
        "var_chol_log_diag": jnp.full((M,), jnp.log(0.1)),
        "var_chol_lower":    jnp.zeros(M * (M - 1) // 2),
    }

## Thompson Sampling

In [ ]:
def thompson_sampling(params, rng_key, n_candidates=2000):
    rng_key, k_grid, k_eps1, k_eps2 = jax.random.split(rng_key, 4)
    candidates = jax.random.uniform(k_grid, shape=(n_candidates, 2), minval=-3.0, maxval=3.0)
    mu, var = svgp_predict(params, candidates, full_cov=False)
    std = jnp.sqrt(jnp.clip(var, 1e-6))
    eps1  = jax.random.normal(k_eps1, shape=(n_candidates,))
    cand1 = candidates[jnp.argmax(mu + std * eps1)]
    eps2  = jax.random.normal(k_eps2, shape=(n_candidates,))
    cand2 = candidates[jnp.argmax(mu + std * eps2)]
    return cand1, cand2, rng_key

## Training Loop

In [ ]:
def train_preference_gp(params, X_i, X_j, y_ij, rng_key,
                        num_epochs=200, lr=0.01, batch_size=64, n_samples=20):
    N_total = y_ij.shape[0]
    opt = optax.adam(lr)
    opt_state = opt.init(params)

    for epoch in range(num_epochs):
        rng_key, perm_key = jax.random.split(rng_key)
        perm = jax.random.permutation(perm_key, N_total)

        for start in range(0, N_total, batch_size):
            idx   = perm[start : start + batch_size]
            X_i_b = X_i[idx]
            X_j_b = X_j[idx]
            y_b   = y_ij[idx]

            def neg_elbo(p):
                val, _ = svgp_elbo_preference(p, X_i_b, X_j_b, y_b, N_total, rng_key, n_samples)
                return -val

            neg_val, grads = jax.value_and_grad(neg_elbo)(params)
            updates, opt_state = opt.update(grads, opt_state)
            params = optax.apply_updates(params, updates)

    return params, rng_key

## Plotting Helpers

In [ ]:
def make_grid(n=80):
    x0 = jnp.linspace(-3, 3, n)
    x1 = jnp.linspace(-3, 3, n)
    X0, X1 = jnp.meshgrid(x0, x1)
    grid = jnp.stack([X0.ravel(), X1.ravel()], axis=-1)
    return grid, X0, X1


def plot_surrogate_and_truth(params, X_i_all, X_j_all, title_suffix, collection_order=None):
    grid, X0, X1 = make_grid(n=80)
    mu_grid, _  = svgp_predict(params, grid, full_cov=False)
    true_grid   = latent_function(grid)
    mu_img   = np.array(mu_grid).reshape(80, 80)
    true_img = np.array(true_grid).reshape(80, 80)
    all_locs = np.array(jnp.concatenate([X_i_all, X_j_all], axis=0))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, img, title in zip(axes,
                               [mu_img, true_img],
                               [f"Surrogate mean {title_suffix}",
                                f"True latent function {title_suffix}"]):
        im = ax.contourf(np.array(X0), np.array(X1), img, levels=30, cmap="viridis")
        fig.colorbar(im, ax=ax)
        if collection_order is not None:
            colors = cm.plasma(np.linspace(0, 1, len(all_locs)))
            for pt, col in zip(all_locs, colors):
                ax.scatter(pt[0], pt[1], color=col, s=20, zorder=3, edgecolors="none")
        else:
            ax.scatter(all_locs[:, 0], all_locs[:, 1],
                       c="white", s=20, zorder=3, edgecolors="none", alpha=0.7)
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
        ax.set_xlabel("x\u2080"); ax.set_ylabel("x\u2081"); ax.set_title(title)
    plt.tight_layout()
    plt.show()

## Verification Checks

In [ ]:
def run_verification():
    print("=" * 60)
    print("Structural verification checks...")
    print("=" * 60)
    rng_key = jax.random.PRNGKey(0)
    passed = 0
    failed = 0

    def check(name, condition, detail=""):
        nonlocal passed, failed
        if condition:
            print(f"  PASS  {name}")
            passed += 1
        else:
            print(f"  FAIL  {name}" + (f" — {detail}" if detail else ""))
            failed += 1

    M = 8
    rng_key, init_key = jax.random.split(rng_key)
    params = init_params_preference(num_inducing=M, rng_key=init_key)

    # CHECK 1: Kernel shapes
    rng_key, k = jax.random.split(rng_key)
    X_a = jax.random.uniform(k, shape=(10, 2))
    rng_key, k = jax.random.split(rng_key)
    X_b = jax.random.uniform(k, shape=(7, 2))
    ls  = jnp.exp(params["log_lengthscale"])
    var = jnp.exp(params["log_variance"])
    K_full = rbf_kernel(X_a, X_b, ls, var, diag=False)
    K_diag = rbf_kernel(X_a, X_a, ls, var, diag=True)
    check("Kernel full shape (10,7)", K_full.shape == (10, 7), str(K_full.shape))
    check("Kernel diag shape (10,)",  K_diag.shape == (10,),   str(K_diag.shape))

    # CHECK 2: Predict output validity
    rng_key, k = jax.random.split(rng_key)
    X_test = jax.random.uniform(k, shape=(20, 2), minval=-3.0, maxval=3.0)
    mu, var_pred = svgp_predict(params, X_test, full_cov=False)
    check("Predict mean finite",  bool(jnp.all(jnp.isfinite(mu))))
    check("Predict variance > 0", bool(jnp.all(var_pred > 0)),
          f"min var = {float(jnp.min(var_pred)):.4e}")

    # CHECK 3: KL divergence non-negative
    kl = svgp_kl_divergence(params)
    check("KL divergence >= 0", bool(kl >= 0), f"kl = {float(kl):.4f}")

    # CHECK 4: ELL finite and negative
    rng_key, k1, k2 = jax.random.split(rng_key, 3)
    X_i_dummy = jax.random.uniform(k1, shape=(15, 2), minval=-3.0, maxval=3.0)
    X_j_dummy = jax.random.uniform(k2, shape=(15, 2), minval=-3.0, maxval=3.0)
    y_dummy   = jnp.array([1,0,1,1,0,1,0,0,1,1,0,1,0,1,0], dtype=jnp.float32)
    rng_key, ell_key = jax.random.split(rng_key)
    ell, _ = svgp_ell_preference(params, X_i_dummy, X_j_dummy, y_dummy, 15, ell_key)
    check("ELL is finite",   bool(jnp.isfinite(ell)), f"ell = {float(ell):.4f}")
    check("ELL is negative", bool(ell < 0),           f"ell = {float(ell):.4f}")

    # CHECK 5: Sigma_ij gives zero variance for identical pairs
    rng_key, k = jax.random.split(rng_key)
    X_same = jax.random.uniform(k, shape=(10, 2), minval=-3.0, maxval=3.0)
    Z   = params["inducing_inputs"]
    L_v = build_var_chol(params["var_chol_log_diag"], params["var_chol_lower"], M)
    K_zz  = rbf_kernel(Z, Z, ls, var)
    L_zz  = stable_cholesky(K_zz)
    X_pairs   = jnp.concatenate([X_same, X_same], axis=0)
    K_z_pairs = rbf_kernel(Z, X_pairs, ls, var)
    alpha_all = jsp.linalg.solve_triangular(L_zz, K_z_pairs, lower=True)
    B_sz = X_same.shape[0]
    alpha_i = alpha_all[:, :B_sz];  alpha_j = alpha_all[:, B_sz:]
    B_i = L_v.T @ alpha_i;          B_j = L_v.T @ alpha_j
    k_ii = rbf_kernel(X_same, X_same, ls, var, diag=True)
    k_ij = rbf_kernel(X_same, X_same, ls, var, diag=True)
    Sigma_ii = k_ii - jnp.sum(alpha_i**2, axis=0) + jnp.sum(B_i**2, axis=0)
    Sigma_ij = k_ij - jnp.sum(alpha_i * alpha_j, axis=0) + jnp.sum(B_i * B_j, axis=0)
    var_diff_same = 2 * Sigma_ii - 2.0 * Sigma_ij
    max_var = float(jnp.max(jnp.abs(var_diff_same)))
    check("Sigma_ij: var_diff ~ 0 for identical pairs", max_var < 1e-4, f"max |var_diff| = {max_var:.2e}")

    # CHECK 6: ELBO increases with training (5 epochs)
    rng_key, init_key2, train_key = jax.random.split(rng_key, 3)
    params_b = init_params_preference(num_inducing=8, rng_key=init_key2)
    elbo_before, _ = svgp_elbo_preference(params_b, X_i_dummy, X_j_dummy, y_dummy, 15, train_key)
    params_a, _    = train_preference_gp(params_b, X_i_dummy, X_j_dummy, y_dummy, train_key,
                                          num_epochs=5, lr=0.01, batch_size=15)
    rng_key, ck = jax.random.split(rng_key)
    elbo_after, _  = svgp_elbo_preference(params_a, X_i_dummy, X_j_dummy, y_dummy, 15, ck)
    check("ELBO increases after 5 training epochs",
          float(elbo_after) > float(elbo_before),
          f"before={float(elbo_before):.2f}, after={float(elbo_after):.2f}")

    # CHECK 7: Thompson sampling stays in domain
    rng_key, ts_key = jax.random.split(rng_key)
    cand1, cand2, _ = thompson_sampling(params_a, ts_key, n_candidates=500)
    in_domain = (jnp.all(cand1 >= -3.0) and jnp.all(cand1 <= 3.0) and
                 jnp.all(cand2 >= -3.0) and jnp.all(cand2 <= 3.0))
    check("Thompson candidates in [-3, 3]^2", bool(in_domain), f"cand1={cand1}, cand2={cand2}")
    check("Thompson candidates shape (2,)",
          cand1.shape == (2,) and cand2.shape == (2,),
          f"shapes: {cand1.shape}, {cand2.shape}")

    # CHECK 8: sample_preferences shapes and binary values
    rng_key, sp_key = jax.random.split(rng_key)
    new_Xi, new_Xj, new_yij = sample_preferences(sp_key, cand1, cand2, 25)
    check("sample_preferences X_i shape (25,2)", new_Xi.shape  == (25, 2), str(new_Xi.shape))
    check("sample_preferences X_j shape (25,2)", new_Xj.shape  == (25, 2), str(new_Xj.shape))
    check("sample_preferences y_ij shape (25,)", new_yij.shape == (25,),   str(new_yij.shape))
    check("sample_preferences y_ij in {0,1}",
          bool(jnp.all((new_yij == 0) | (new_yij == 1))))

    print("-" * 60)
    print(f"  {passed} passed, {failed} failed")
    print("=" * 60)
    if failed > 0:
        raise RuntimeError(f"{failed} structural check(s) failed.")


def run_output_correctness_checks():
    print("=" * 60)
    print("Output correctness checks...")
    print("=" * 60)
    rng_key = jax.random.PRNGKey(42)
    passed = 0
    failed = 0

    def check(name, condition, detail=""):
        nonlocal passed, failed
        if condition:
            print(f"  PASS  {name}")
            passed += 1
        else:
            print(f"  FAIL  {name}" + (f" — {detail}" if detail else ""))
            failed += 1

    rng_key, init_key = jax.random.split(rng_key)
    params = init_params_preference(num_inducing=10, rng_key=init_key)
    ls  = jnp.exp(params["log_lengthscale"])
    var = jnp.exp(params["log_variance"])
    rng_key, ka, kb = jax.random.split(rng_key, 3)
    X_a = jax.random.uniform(ka, shape=(8, 2), minval=-3.0, maxval=3.0)
    X_b = jax.random.uniform(kb, shape=(8, 2), minval=-3.0, maxval=3.0)

    # CHECK A: Kernel symmetry
    K_ab = rbf_kernel(X_a, X_b, ls, var)
    K_ba = rbf_kernel(X_b, X_a, ls, var)
    sym_err = float(jnp.max(jnp.abs(K_ab - K_ba.T)))
    check("Kernel symmetry K(A,B) == K(B,A).T", sym_err < 1e-6, f"max err = {sym_err:.2e}")

    # CHECK B: Kernel diagonal equals variance
    K_aa_diag = rbf_kernel(X_a, X_a, ls, var, diag=True)
    diag_err  = float(jnp.max(jnp.abs(K_aa_diag - var)))
    check("Kernel diagonal equals variance", diag_err < 1e-6, f"max err = {diag_err:.2e}")

    # CHECK B2: diag=True matches diag of full matrix
    K_aa_full = rbf_kernel(X_a, X_a, ls, var, diag=False)
    diag2_err = float(jnp.max(jnp.abs(K_aa_diag - jnp.diag(K_aa_full))))
    check("Kernel diag=True matches diag of full matrix", diag2_err < 1e-6, f"max err = {diag2_err:.2e}")

    # CHECK C: ELL at f_diff=0 matches -log(2)*N
    N = 20
    rng_key, kx = jax.random.split(rng_key)
    X_same = jax.random.uniform(kx, shape=(N, 2), minval=-3.0, maxval=3.0)
    y_half = jnp.array([float(i % 2) for i in range(N)])
    rng_key, ell_key = jax.random.split(rng_key)
    ell_same, _ = svgp_ell_preference(params, X_same, X_same, y_half, N, ell_key, n_samples=200)
    expected_ell = -jnp.log(2.0) * N
    ell_err = float(jnp.abs(ell_same - expected_ell))
    check("ELL at f_diff=0 \u2248 -log(2)*N", ell_err < 0.5,
          f"got {float(ell_same):.3f}, expected {float(expected_ell):.3f}")

    # CHECK D: ELL higher when model is certain and correct
    Z = params["inducing_inputs"]
    large_m = jnp.zeros(10).at[0].set(10.0)
    params_certain = {**params, "variational_mean": large_m}
    X_i_certain = jnp.tile(Z[0], (10, 1))
    rng_key, kfar = jax.random.split(rng_key)
    X_j_certain = jax.random.uniform(kfar, shape=(10, 2), minval=-2.9, maxval=-2.0)
    y_certain = jnp.ones(10)
    rng_key, ell_key2 = jax.random.split(rng_key)
    ell_certain, _ = svgp_ell_preference(params_certain, X_i_certain, X_j_certain,
                                          y_certain, 10, ell_key2, n_samples=100)
    rng_key, ell_key3 = jax.random.split(rng_key)
    ell_random, _ = svgp_ell_preference(params, X_i_certain, X_j_certain,
                                         y_certain, 10, ell_key3, n_samples=100)
    check("ELL is higher when model is certain and correct",
          float(ell_certain) > float(ell_random),
          f"certain={float(ell_certain):.3f}, random_init={float(ell_random):.3f}")

    # CHECK E: Posterior mean reflects preference direction after training
    rng_key, ka2, kb2 = jax.random.split(rng_key, 3)
    X_i_pref = jnp.clip(jax.random.normal(ka2, shape=(30, 2)) * 0.3 + jnp.array([2.0, 2.0]), -3.0, 3.0)
    X_j_pref = jnp.clip(jax.random.normal(kb2, shape=(30, 2)) * 0.3 + jnp.array([-2.0, -2.0]), -3.0, 3.0)
    y_pref = jnp.ones(30)
    rng_key, init_pref, train_pref = jax.random.split(rng_key, 3)
    params_pref = init_params_preference(num_inducing=10, rng_key=init_pref)
    params_pref, _ = train_preference_gp(params_pref, X_i_pref, X_j_pref, y_pref,
                                          train_pref, num_epochs=50, lr=0.01, batch_size=30)
    region_A = jnp.array([[1.5, 1.5], [2.0, 2.0], [1.8, 2.0]])
    region_B = jnp.array([[-1.5, -1.5], [-2.0, -2.0], [-1.8, -2.0]])
    mu_A, _ = svgp_predict(params_pref, region_A, full_cov=False)
    mu_B, _ = svgp_predict(params_pref, region_B, full_cov=False)
    check("Surrogate mean higher in preferred region after training",
          float(jnp.mean(mu_A)) > float(jnp.mean(mu_B)),
          f"mean(A)={float(jnp.mean(mu_A)):.3f}, mean(B)={float(jnp.mean(mu_B)):.3f}")

    # CHECK F: Thompson sampling explores the space
    all_cands = []
    for i in range(20):
        rng_key, ts_k = jax.random.split(rng_key)
        c1, c2, _ = thompson_sampling(params_pref, ts_k, n_candidates=500)
        all_cands.extend([np.array(c1), np.array(c2)])
    all_cands = np.stack(all_cands)
    std_x0 = float(np.std(all_cands[:, 0]))
    std_x1 = float(np.std(all_cands[:, 1]))
    check("Thompson sampling explores x\u2080 axis (std > 0.5)", std_x0 > 0.5, f"std = {std_x0:.3f}")
    check("Thompson sampling explores x\u2081 axis (std > 0.5)", std_x1 > 0.5, f"std = {std_x1:.3f}")

    # CHECK G: Dataset grows by exactly 25 after one round
    rng_key, sp_k = jax.random.split(rng_key)
    N_before = int(y_pref.shape[0])
    new_Xi, new_Xj, new_yij = sample_preferences(sp_k, all_cands[0], all_cands[1], 25)
    y1 = jnp.concatenate([y_pref, new_yij])
    check("Dataset grows by 25 after one round", int(y1.shape[0]) == N_before + 25,
          f"{N_before} \u2192 {int(y1.shape[0])}")
    check("All appended y values in {0,1}",
          bool(jnp.all((new_yij == 0) | (new_yij == 1))))

    print("-" * 60)
    print(f"  {passed} passed, {failed} failed")
    print("=" * 60)
    if failed > 0:
        raise RuntimeError(f"{failed} output correctness check(s) failed.")

In [ ]:
run_verification()
run_output_correctness_checks()

## Bayesian Optimisation

In [ ]:
rng_key = jax.random.PRNGKey(1)

print("Generating initial preference data...")
X_i, X_j, y_ij, _, _ = generate_initial_preference_data(seed=7)
X_i  = jax.device_put(X_i,  _device)
X_j  = jax.device_put(X_j,  _device)
y_ij = jax.device_put(y_ij, _device)
print(f"  Initial pairs: {y_ij.shape[0]}")

In [ ]:
print("Training initial surrogate model...")
rng_key, init_key = jax.random.split(rng_key)
params = init_params_preference(num_inducing=15, rng_key=init_key)
params, rng_key = train_preference_gp(
    params, X_i, X_j, y_ij, rng_key, num_epochs=300, lr=0.01, batch_size=32
)
print("  Done.")

In [ ]:
plot_surrogate_and_truth(params, X_i, X_j, "(initial)")

In [ ]:
print("Starting Bayesian Optimisation loop (50 rounds)...")
for round_idx in range(50):
    rng_key, ts_key = jax.random.split(rng_key)
    cand1, cand2, _ = thompson_sampling(params, ts_key, n_candidates=2000)

    rng_key, sp_key = jax.random.split(rng_key)
    new_Xi, new_Xj, new_yij = sample_preferences(sp_key, cand1, cand2,
                                                   preference_samples_per_location=25)
    X_i  = jnp.concatenate([X_i,  new_Xi])
    X_j  = jnp.concatenate([X_j,  new_Xj])
    y_ij = jnp.concatenate([y_ij, new_yij])

    params, rng_key = train_preference_gp(
        params, X_i, X_j, y_ij, rng_key, num_epochs=100, lr=0.005, batch_size=64
    )

    print(f"  Round {round_idx + 1}/50  |  total pairs: {y_ij.shape[0]}  |  "
          f"cand1=[{float(cand1[0]):.2f}, {float(cand1[1]):.2f}]  "
          f"cand2=[{float(cand2[0]):.2f}, {float(cand2[1]):.2f}]")

print("BO loop complete.")

In [ ]:
all_locs_ordered = np.array(jnp.concatenate([X_i, X_j], axis=0))
plot_surrogate_and_truth(params, X_i, X_j, "(after 50 rounds)",
                          collection_order=all_locs_ordered)

In [ ]:
# CHECK H: Surrogate argmax should be near the true peak at (1.2, -1.0)
grid, _, _ = make_grid(n=200)
mu_grid, _ = svgp_predict(params, grid, full_cov=False)
best_x     = grid[jnp.argmax(mu_grid)]
true_peak  = jnp.array([1.2, -1.0])
dist       = float(jnp.sqrt(jnp.sum((best_x - true_peak) ** 2)))
print(f"Estimated optimum: x = [{float(best_x[0]):.3f}, {float(best_x[1]):.3f}]")
print(f"True optimum near: x = [1.2, -1.0]")
print(f"Distance to true peak: {dist:.3f}")
if dist < 1.0:
    print("  CHECK H PASS  Surrogate argmax within 1.0 of true peak")
else:
    print(f"  CHECK H FAIL  Distance {dist:.3f} >= 1.0 — BO may not have converged")